# Standardised Feature Engineering Notebook

This notebook **standardises** the **feature engineering** workflow for **historical stablecoin datasets**. Users should **update** the file paths for both *data input and output* before running the notebook. The cleaned dataset is saved in **Parquet** format to *preserve variable types* and ensure consistency across downstream analysis. This workflow is designed to be easily replicated for other stablecoins by changing the relevant file paths only.

The predictive features are constructed to closely follow the framework proposed in Lee, Chiu, and Hsieh (2025), **Stablecoin depegging risk prediction**, published in the Pacific-Basin Finance Journal. This helps maintain consistency with the literature while allowing the feature engineering process to be applied systematically across different stablecoin datasets.

In [1]:
import pandas as pd
import numpy as np

In [2]:
stablecoin = "usdt" # or "usdc", "usdt", "pax", "dai", "ustd"
df = pd.read_parquet(f"clean_data/{stablecoin}_clean.parquet")
df.head()

,symbol,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,timestamp,timeOpen,timeClose,timeHigh,timeLow,open,high,low,close,volume,marketCap,circulatingSupply
0,USDT,0,0,0,0,0,0,0,2020-11-25 23:59:59,2020-11-25,2020-11-25 23:59:59,2020-11-25 16:35:12,2020-11-25 02:01:08,0.999805,1.001169,0.999456,1.000053,8.688931e+10,1.865547e+10,1.865448e+10
1,USDT,0,0,0,0,0,0,0,2020-11-26 23:59:59,2020-11-26,2020-11-26 23:59:59,2020-11-26 18:56:08,2020-11-26 08:28:09,1.000021,1.002408,0.999624,1.002035,1.146168e+11,1.899623e+10,1.895766e+10
2,USDT,0,0,0,0,0,0,0,2020-11-27 23:59:59,2020-11-27,2020-11-27 23:59:59,2020-11-27 01:30:10,2020-11-27 15:30:13,1.001955,1.002112,1.000664,1.001200,7.101232e+10,1.898040e+10,1.895766e+10
3,USDT,0,0,0,0,0,0,0,2020-11-28 23:59:59,2020-11-28,2020-11-28 23:59:59,2020-11-28 02:11:08,2020-11-28 16:13:08,1.001181,1.001583,1.000898,1.001001,5.962222e+10,1.905734e+10,1.903829e+10
4,USDT,0,0,0,0,0,0,0,2020-11-29 23:59:59,2020-11-29,2020-11-29 23:59:59,2020-11-29 06:08:08,2020-11-29 20:50:12,1.000981,1.001338,1.000599,1.000890,5.628394e+10,1.912689e+10,1.910989e+10


## Feature Engineering

### Rate of Change in Trading Price and Volume

In [3]:
# price percent changes
df["percent_change_24h"] = df["close"].pct_change(1) * 100
df["percent_change_7d"]  = df["close"].pct_change(7) * 100
df["percent_change_30d"] = df["close"].pct_change(30) * 100

# volume percent changes
df["volume_percent_change_24h"] = df["volume"].pct_change(1) * 100
df["volume_percent_change_7d"]  = df["volume"].pct_change(7) * 100
df["volume_percent_change_30d"] = df["volume"].pct_change(30) * 100

### Rate of Change in Market Information

In [4]:
def ln_past_over_today(x, k):
    return np.log(x.shift(k) / x)

# market cap log-changes
df["market_cap_percent_change_24h"] = ln_past_over_today(df["marketCap"], 1)
df["market_cap_percent_change_7d"]  = ln_past_over_today(df["marketCap"], 7)
df["market_cap_percent_change_30d"] = ln_past_over_today(df["marketCap"], 30)

# circulating supply log-changes
df["circulating_supply_percent_change_24h"] = ln_past_over_today(df["circulatingSupply"], 1)
df["circulating_supply_percent_change_7d"]  = ln_past_over_today(df["circulatingSupply"], 7)
df["circulating_supply_percent_change_30d"] = ln_past_over_today(df["circulatingSupply"], 30)

### Volatility Indicators

In [5]:
# avoid log(0) issues
eps = 1e-12
O = df["open"].clip(lower=eps)
H = df["high"].clip(lower=eps)
L = df["low"].clip(lower=eps)
C = df["close"].clip(lower=eps)

df["realized_daily_volatility"] = np.sqrt(
    (np.log(H / O) * np.log(H / C)) + (np.log(L / O) * np.log(L / C))
)

In [6]:
df["peg_error"] = df["close"] - 1
df["abs_peg_error"] = df["peg_error"].abs()

df["price_deviation_5d"]  = np.sqrt((df["peg_error"]**2).rolling(5).mean())
df["price_deviation_30d"] = np.sqrt((df["peg_error"]**2).rolling(30).mean())

downside = np.minimum(df["peg_error"], 0.0)
df["downward_price_deviation_5d"]  = np.sqrt((downside**2).rolling(5).mean())
df["downward_price_deviation_30d"] = np.sqrt((downside**2).rolling(30).mean())


### Cleanup
To ensure model compatibility, infinite values arising from logarithmic transformations and ratio-based features are first replaced with missing values. Observations containing missing values—primarily introduced by rolling-window and lagged feature construction—are subsequently removed. This ensures that all downstream analyses, including PCA and machine learning models, are performed on a complete and consistent dataset.

In [7]:
# replace inf/-inf from logs with NaN
df = df.replace([np.inf, -np.inf], np.nan)

# drop NaNs (ie for early rows due to rolling/shift)
df = df.dropna().reset_index(drop=True)
df.head()

,symbol,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,timestamp,timeOpen,...,circulating_supply_percent_change_24h,circulating_supply_percent_change_7d,circulating_supply_percent_change_30d,realized_daily_volatility,peg_error,abs_peg_error,price_deviation_5d,price_deviation_30d,downward_price_deviation_5d,downward_price_deviation_30d
0,USDT,0,0,0,0,0,0,0,2020-12-25 23:59:59,2020-12-25,...,-0.007327,-0.031696,-0.103282,0.000853,0.000168,0.000168,0.000323,0.000726,0.000314,0.000157
1,USDT,0,0,0,0,0,0,0,2020-12-26 23:59:59,2020-12-26,...,-0.001449,-0.029584,-0.088610,0.000555,-0.001519,0.001519,0.000748,0.000682,0.000745,0.000319
2,USDT,0,0,0,0,0,0,0,2020-12-27 23:59:59,2020-12-27,...,-0.001881,-0.022061,-0.090491,0.002211,-0.001146,0.001146,0.000893,0.000679,0.000889,0.000381
3,USDT,0,0,0,0,0,0,0,2020-12-28 23:59:59,2020-12-28,...,-0.003257,-0.025072,-0.089504,0.001482,-0.001397,0.001397,0.001059,0.000702,0.001056,0.000459
4,USDT,0,0,0,0,0,0,0,2020-12-29 23:59:59,2020-12-29,...,-0.001008,-0.021695,-0.086758,0.000508,-0.001410,0.001410,0.001232,0.000730,0.001230,0.000526


## Combining Sentiment Indicators

In [8]:
# create a daily date key (so merges are clean)
df["date"] = df["timestamp"].dt.date
df["date"] = pd.to_datetime(df["date"])
df.head()

,symbol,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,timestamp,timeOpen,...,circulating_supply_percent_change_7d,circulating_supply_percent_change_30d,realized_daily_volatility,peg_error,abs_peg_error,price_deviation_5d,price_deviation_30d,downward_price_deviation_5d,downward_price_deviation_30d,date
0,USDT,0,0,0,0,0,0,0,2020-12-25 23:59:59,2020-12-25,...,-0.031696,-0.103282,0.000853,0.000168,0.000168,0.000323,0.000726,0.000314,0.000157,2020-12-25
1,USDT,0,0,0,0,0,0,0,2020-12-26 23:59:59,2020-12-26,...,-0.029584,-0.088610,0.000555,-0.001519,0.001519,0.000748,0.000682,0.000745,0.000319,2020-12-26
2,USDT,0,0,0,0,0,0,0,2020-12-27 23:59:59,2020-12-27,...,-0.022061,-0.090491,0.002211,-0.001146,0.001146,0.000893,0.000679,0.000889,0.000381,2020-12-27
3,USDT,0,0,0,0,0,0,0,2020-12-28 23:59:59,2020-12-28,...,-0.025072,-0.089504,0.001482,-0.001397,0.001397,0.001059,0.000702,0.001056,0.000459,2020-12-28
4,USDT,0,0,0,0,0,0,0,2020-12-29 23:59:59,2020-12-29,...,-0.021695,-0.086758,0.000508,-0.001410,0.001410,0.001232,0.000730,0.001230,0.000526,2020-12-29


### Load Fear & Greed

In [9]:
fg = pd.read_csv("raw_data/fear_greed_index.csv")

fg["date"] = pd.to_datetime(fg["date"], errors="coerce")  
fg = fg.dropna(subset=["date"]).sort_values("date")
fg["date"] = fg["date"].dt.tz_localize(None).dt.normalize()

# rename columns for clarity
fg = fg.rename(columns={
    "value": "fear_greed_index",
    "value_classification": "fear_greed_index_category",
    "date": "date"
})

### Load Fed Funds Rate

In [10]:
ff = pd.read_csv("raw_data/fed_funds_rate.csv")

ff["date"] = pd.to_datetime(ff["date"], errors="coerce")
ff = ff.dropna(subset=["date"]).sort_values("date")
ff["date"] = ff["date"].dt.tz_localize(None).dt.normalize()

### Merge

In [11]:
print(df["date"].dtype)
print(fg["date"].dtype)
print(ff["date"].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]


In [12]:
final_df = df.merge(fg, on="date", how="left").merge(ff, on="date", how="left")
final_df = final_df.drop(columns=["date"])
final_df.head()

,symbol,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,timestamp,timeOpen,...,realized_daily_volatility,peg_error,abs_peg_error,price_deviation_5d,price_deviation_30d,downward_price_deviation_5d,downward_price_deviation_30d,fear_greed_index,fear_greed_index_category,fed_funds_rate
0,USDT,0,0,0,0,0,0,0,2020-12-25 23:59:59,2020-12-25,...,0.000853,0.000168,0.000168,0.000323,0.000726,0.000314,0.000157,94.0,Extreme Greed,0.09
1,USDT,0,0,0,0,0,0,0,2020-12-26 23:59:59,2020-12-26,...,0.000555,-0.001519,0.001519,0.000748,0.000682,0.000745,0.000319,93.0,Extreme Greed,0.09
2,USDT,0,0,0,0,0,0,0,2020-12-27 23:59:59,2020-12-27,...,0.002211,-0.001146,0.001146,0.000893,0.000679,0.000889,0.000381,91.0,Extreme Greed,0.09
3,USDT,0,0,0,0,0,0,0,2020-12-28 23:59:59,2020-12-28,...,0.001482,-0.001397,0.001397,0.001059,0.000702,0.001056,0.000459,92.0,Extreme Greed,0.09
4,USDT,0,0,0,0,0,0,0,2020-12-29 23:59:59,2020-12-29,...,0.000508,-0.001410,0.001410,0.001232,0.000730,0.001230,0.000526,91.0,Extreme Greed,0.09


---

## Save

In [13]:
print(final_df.shape)
final_df.dtypes

(1911, 42)


symbol                                           object
depeg                                             int64
depeg_future_1d                                   int64
depeg_future_3d                                   int64
depeg_future_5d                                   int64
depeg_future_7d                                   int64
depeg_future_14d                                  int64
depeg_future_30d                                  int64
timestamp                                datetime64[ns]
timeOpen                                 datetime64[ns]
timeClose                                datetime64[ns]
timeHigh                                 datetime64[ns]
timeLow                                  datetime64[ns]
open                                            float64
high                                            float64
low                                             float64
close                                           float64
volume                                          

In [14]:
final_df.to_parquet(f"clean_data/{stablecoin}_final.parquet")